# Module 3: Model Optimization Techniques

The difference between a good model and a great model often comes down to optimization. This notebook covers three approaches to hyperparameter tuning and compares ensemble methods.

**Key insight:** Random Search typically gets 95% of the way to optimal in 10% of the time Grid Search takes. Bayesian optimization (Optuna) is smarter about which combinations to try.

## 1. Setup

We create a moderately complex classification problem — 3,000 samples, 20 features, 30% minority class. This is realistic enough that hyperparameter choices actually matter.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import optuna
from optuna.samplers import TPESampler
import time
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

X, y = make_classification(
    n_samples=3000, n_features=20, n_informative=12,
    n_redundant=4, weights=[0.7, 0.3], random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

## 2. Grid Search — The Brute Force Approach

**How it works:** Try every combination of hyperparameters in a predefined grid.

**Pros:** Guaranteed to find the best combination within the grid.
**Cons:** Exponentially expensive. A grid of 4 params with 3 values each = 81 combinations x 3-fold CV = 243 model fits.

**When to use:** Small search spaces, cheap models, when you need certainty.

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0],
}

total_combos = np.prod([len(v) for v in param_grid.values()])
print(f"Grid combinations: {total_combos}")
print(f"With 3-fold CV: {total_combos * 3} model fits\n")

start = time.time()
gs = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid, cv=3, scoring="roc_auc", n_jobs=-1,
)
gs.fit(X_train, y_train)
gs_time = time.time() - start

gs_score = roc_auc_score(y_test, gs.predict_proba(X_test)[:, 1])
print(f"Grid Search: AUC={gs_score:.4f} ({gs_time:.1f}s)")
print(f"Best params: {gs.best_params_}")

## 3. Random Search — Smarter Than It Sounds

**Key insight from Bergstra & Bengio (2012):** Random search is more efficient than grid search because not all hyperparameters are equally important. Grid search wastes time testing combinations of unimportant parameters, while random search covers more of the important dimensions.

With 30 random samples from a broader space, we often match or beat grid search.

In [ ]:
param_dist = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 4, 5, 6, 7, 8],
    "learning_rate": [0.005, 0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_samples_leaf": [1, 2, 5, 10],
}

start = time.time()
rs = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_dist, n_iter=30, cv=3, scoring="roc_auc", random_state=42, n_jobs=-1,
)
rs.fit(X_train, y_train)
rs_time = time.time() - start

rs_score = roc_auc_score(y_test, rs.predict_proba(X_test)[:, 1])
print(f"Random Search: AUC={rs_score:.4f} ({rs_time:.1f}s)")
print(f"Best params: {rs.best_params_}")

## 4. Bayesian Optimization with Optuna

**How it works:** Instead of blind search, Optuna uses past trial results to build a probabilistic model of which hyperparameter regions are promising (TPE — Tree-structured Parzen Estimator). Each new trial is chosen to maximize expected improvement.

**Why Optuna over other Bayesian tools?** Optuna supports:
- Pruning bad trials early (saves compute)
- Conditional parameters (parameter B only matters if parameter A > threshold)
- Built-in visualization and analysis

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
    }
    model = GradientBoostingClassifier(**params, random_state=42)
    model.fit(X_train, y_train)
    return roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

start = time.time()
study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50)
op_time = time.time() - start

print(f"Optuna: AUC={study.best_value:.4f} ({op_time:.1f}s)")
print(f"Best params: {study.best_params}")

## 5. Comparison: Which Approach Wins?

The real comparison isn't just accuracy — it's accuracy per unit of compute time.

In [ ]:
print("HYPERPARAMETER TUNING COMPARISON")
print("=" * 55)
header = f"{'Method':<20} {'AUC':>8} {'Time':>8} {'Trials':>8}"
print(header)
print("-" * 55)
grid_combos = int(np.prod([len(v) for v in param_grid.values()]))
print(f"{'Grid Search':<20} {gs_score:>8.4f} {gs_time:>7.1f}s {grid_combos:>8}")
print(f"{'Random Search':<20} {rs_score:>8.4f} {rs_time:>7.1f}s {30:>8}")
print(f"{'Optuna (Bayesian)':<20} {study.best_value:>8.4f} {op_time:>7.1f}s {50:>8}")

print(f"\nAnalysis:")
print(f"  Grid Search explored a LIMITED space exhaustively")
print(f"  Random Search explored a BROADER space with fewer trials")
print(f"  Optuna explored INTELLIGENTLY - focusing on promising regions")

## 6. Ensemble Methods Comparison

Ensembles combine multiple models to reduce variance (bagging) or bias (boosting). Let's compare the major approaches.

In [ ]:
from sklearn.ensemble import (
    BaggingClassifier, RandomForestClassifier, AdaBoostClassifier,
    GradientBoostingClassifier, StackingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
import xgboost as xgb

ensembles = {
    "Bagging (Decision Tree)": BaggingClassifier(
        estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42,
    ),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=200, max_depth=5, use_label_encoder=False,
        eval_metric="logloss", random_state=42,
    ),
}

header = f"{'Model':<30} {'ROC-AUC':>8} {'CV AUC (mean +/- std)'}"
print(header)
print("=" * 65)

for name, model in ensembles.items():
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    cv = cross_val_score(model, X_train, y_train, cv=3, scoring="roc_auc")
    print(f"{name:<30} {auc:>8.4f} {cv.mean():.4f} +/- {cv.std():.4f}")

## Key Takeaways

1. **Random Search > Grid Search** for most practical purposes — broader coverage, less compute
2. **Optuna (Bayesian)** is best when trials are expensive — it learns which regions to explore
3. **XGBoost/Gradient Boosting** typically wins on tabular data — start there as your strong baseline
4. **Stacking** can squeeze out extra performance but adds complexity — use when 0.1% AUC matters
5. **Always cross-validate** — a single test score can be lucky; CV gives confidence intervals